In [ ]:
import torch

# Check if GPU or MPS is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU is enabled: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS GPU is enabled.")
else:
    raise OSError(
        "No GPU or MPS device found. Please check your environment and ensure GPU or MPS support is configured."
    )

CUDA GPU is enabled: NVIDIA GeForce GTX 1050 Ti


In [3]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
import ollama

c:\Users\krish\Work\projects\esg-disclosure-classifier\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
converter = DocumentConverter()
chunker = HybridChunker()

doc = converter.convert(r"../airbus_report_of_the_board_of_directors_2024.pdf").document

texts = [chunk.text for chunk in chunker.chunk(doc)]

2025-11-12 23:44:51,229 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-12 23:44:53,410 - INFO - Going to convert document batch...
2025-11-12 23:44:53,412 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-12 23:44:53,431 - INFO - Loading plugin 'docling_defaults'
2025-11-12 23:44:53,438 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-12 23:44:53,461 - INFO - Loading plugin 'docling_defaults'
2025-11-12 23:44:53,475 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-11-12 23:44:53,618 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2025-11-12 23:44:53,620 - INFO - easyocr cannot be used because it is not installed.
2025-11-12 23:44:54,391 - INFO - Accelerator device: 'cuda:0'
[INFO] 2025-11-12 23:44:54,462 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-11-12 23:44:54,564 [RapidOCR] download_file.py:60: 

time reduced to 13 mins with GPU

In [7]:
with open('converted_doc.txt', 'w', encoding='utf8') as file:
    file.writelines(texts)

In [8]:
def embed(text_chunk: str, model_name='mxbai-embed-large'):
    return ollama.embed(
        model=model_name,
        input=text_chunk
    )['embeddings'][0]

In [9]:
from pymilvus import Collection, connections, DataType, FieldSchema, CollectionSchema, utility
import os

MILVUS_HOST = os.getenv('MILVUS_HOST', "localhost")
MILVUS_PORT = os.getenv('MILVUS_PORT', '19530')
COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'report_chunks')
EMBED_DIM = os.getenv('EMBED_DIM', 1024)   

test_embeddings = ollama.embed(
  model='nomic-embed-text:v1.5',
  input='Llamas are members of the camelid family',
)

EMBED_DIM = len(test_embeddings.embeddings[0])
print(EMBED_DIM)

model_name = 'nomic-embed-text:v1.5'

class MilvusManager:
    _instance = None
    _collection = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(MilvusManager, cls).__new__(cls)
            cls._instance._connect()
            cls._instance._init_collection()
        return cls._instance

    def _connect(self):
        '''Establish a single connection to Milvus.'''
        connections.connect("default", host=MILVUS_HOST, port=MILVUS_PORT)
        print(connections.list_connections())

    def _init_collection(self):
        '''Initialize or load collection schema once.'''
        fields = [
            FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
            FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=EMBED_DIM),
            FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=2000)
        ]
        schema = CollectionSchema(fields, description="Report text chunks with embeddings")

        try:
            self._collection = Collection(COLLECTION_NAME)
        except Exception:
            self._collection = Collection(name=COLLECTION_NAME, schema=schema)

        # Create index if not already present
        if not self._collection.has_index():
            self._collection.create_index(
                field_name="embedding",
                index_params={"metric_type": "COSINE", "index_type": "IVF_FLAT", "params": {"nlist": 128}}
            )

    def get_collection(self) -> Collection:
        '''Return the active collection object.'''
        return self._collection
    
    def semantic_search(self, query: str, top_k: int = 5):
        '''
        
        '''
        vector = embed(query, model_name)

        collection = self.get_collection()
        collection.load()

        results = collection.search(
            data=[vector],
            anns_field="embedding",
            param={"metric_type": "COSINE", "params": {"nprobe": 10}},
            limit=top_k,
            output_fields=["text"]
        )

        return [hit.entity.get("text") for hit in results[0]]

2025-11-13 00:03:13,793 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


768


In [10]:
model_name = 'nomic-embed-text:v1.5'

embeddings = [embed(chunk, model_name) for chunk in texts]

milvus = MilvusManager()
collection = milvus.get_collection()

data = [embeddings, texts]  
collection.insert(data)
collection.flush()

2025-11-13 00:06:51,423 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 00:06:51,522 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 00:06:51,606 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 00:06:51,685 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 00:06:51,765 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 00:06:51,848 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 00:06:51,930 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 00:06:52,004 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 00:06:52,082 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 00:06:52,129 - INFO - HTTP Request: POST http://127.0.0.1:1143

[('default', <pymilvus.client.grpc_handler.GrpcHandler object at 0x000001EBB3847A10>)]


E5-5_10
37d:

37.

The undertaking shall disclose the following information on its total amount of waste from its own operations, in tonnes or kilogrammes:
(a) the total amount of waste generated ;

(b) the total amount by weight diverted from disposal, with a breakdown between hazardous waste and non-hazardous waste and a breakdown by the following recovery operation types:

i. preparation for reuse;

ii. recycling ; and

iii. other recovery operations.

(c) the amount by weight directed to disposal by waste treatment type and the total amount summing all three types, with a breakdown between hazardous waste and non-hazardous waste. The waste treatment types to be disclosed are:

i. incineration ;

ii. landfill; and

iii. other disposal operations;

(d) the total amount and percentage of non-recycled waste ( 91 ) . 


In [12]:
search_terms = [
    "Total waste generated and diverted from disposal by type and recovery operation",
    "Hazardous and non-hazardous waste breakdown by recycling, reuse, and recovery",
    "Waste directed to disposal by treatment type such as incineration, landfill, or other",
    "Total and percentage of non-recycled waste disclosure requirement",
    "Waste generation and treatment data reported in tonnes or kilograms under own operations",

]

search_terms_keywords = [
    "total waste generated tonnes kilograms",
    "hazardous non-hazardous waste recovery reuse recycling",
    "waste disposal incineration landfill other operations",
    "non-recycled waste total percentage",
    "waste treatment type diversion disposal breakdown",
    ]

In [13]:
context = milvus.semantic_search(query=search_terms[0])

2025-11-13 00:59:26,439 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


In [14]:
context

["Total amount of waste generated in Company's own operations, Unit = tons. Total amount of waste generated in Company's own operations, 2024 = 229,389. Total amount of waste diverted from disposal by recovery operations, Unit = tons. Total amount of waste diverted from disposal by recovery operations, 2024 = 207,469. Total amount of hazardous waste diverted from disposal by recovery operations, Unit = tons. Total amount of hazardous waste diverted from disposal by recovery operations, 2024 = 15,236. Of which, amount going to recycling, Unit = tons. Of which, amount going to recycling, 2024 = 6,828. Of which, amount going to energy recovery, Unit = tons. Of which, amount going to energy recovery, 2024 = 5,523. Of which, amount going to any other recovery operation (pre-treatment / temporary status), Unit = tons. Of which, amount going to any other recovery operation (pre-treatment / temporary status), 2024 = 2,885. Total amount of non hazardous waste diverted from disposal by recovery 